# Finetuning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-deepmind/gemma/blob/main/colabs/finetuning.ipynb)

This is an example on fine-tuning Gemma. For an example on how to run a pre-trained Gemma model, see the [sampling](https://gemma-llm.readthedocs.io/en/latest/sampling.html) tutorial.

To fine-tune Gemma, we use the [kauldron](https://kauldron.readthedocs.io/en/latest/) library which abstract most of the boilerplate (checkpoint management, training loop, evaluation, metric reporting, sharding,...).


In [2]:

# !pip install --upgrade "jax[cuda12]"

In [3]:
import jax
import jaxlib
print(f"Jax backend: {jax.default_backend()}")

Jax backend: gpu


In [4]:
# Common imports
import os
import optax
import treescope

# Gemma imports
from kauldron import kd
from gemma import gm

By default, Jax do not utilize the full GPU memory, but this can be overwritten. See [GPU memory allocation](https://docs.jax.dev/en/latest/gpu_memory_allocation.html):

In [5]:
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

## Data pipeline

First create the tokenizer, as it's required in the data pipeline.

In [4]:
from sklearn.model_selection import train_test_split
import json
f = open('classification_results.json', 'r')
classification_results = json.load(f)
f.close()
train, test = train_test_split(classification_results, test_size=0.1)

In [5]:
test

[{'context': "270097608326905857:  I am not familiar with other block chains, so no idea what approve does in that context\n375962561834909696:  I understand. And if I want to calculate the gas cost for a transaction myself, is that possible?\n270097608326905857:  it's usually done by dryRunning the transaction.\n\nYou can see how its done here:\n\nhttps://github.com/MystenLabs/sui/blob/main/sdk/typescript/src/transactions/json-rpc-resolver.ts#L64-L103\n270097608326905857:  its not possible to model it perfectly though, because there are some aspects of transactions that can change depending on the on-chain state that will affect the final gas cost",
  'message': '375962561834909696: okay, thank u so much)\n',
  'predicted_class': 'Topic'},
 {'context': "983010262003101766:  hey guys! Does anyone know any updates about SuiPlay? I bought the console in preorder on September and all I can do is to login with my email and see the NFT. When it will be possible to bind this NFT to Sui addre

In [6]:
tokenizer = gm.text.Gemma3Tokenizer()

tokenizer.encode('This is an example sentence', add_bos=True)

2025-08-07 20:26:07.580864: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754598367.642001   42034 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754598367.655117   42034 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754598367.697789   42034 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754598367.697927   42034 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754598367.697930   42034 computation_placer.cc:177] computation placer alr

[<_Gemma3SpecialTokens.BOS: 2>, 2094, 563, 614, 2591, 13315]

First we need a data pipeline. Multiple pipelines are supported including:

* [HuggingFace](https://kauldron.readthedocs.io/en/latest/api/kd/data/py/HuggingFace.html)
* [TFDS](https://kauldron.readthedocs.io/en/latest/api/kd/data/py/Tfds.html)
* [Json](https://kauldron.readthedocs.io/en/latest/api/kd/data/py/Json.html)
* ...

It's quite simple to add your own data, or to create mixtures from multiple sources. See the [pipeline documentation](https://kauldron.readthedocs.io/en/latest/data_py.html).

We use `transforms` to customize the data pipeline, this includes:

* Tokenizing the inputs (with `gm.data.Tokenize`)
* Creating the model inputs (with `gm.data.Tokenize`))
* Adding padding (with `gm.data.Pad`) (required to batch inputs with different lengths)

Note that in practice, you can combine multiple transforms into a higher level transform. See the `gm.data.ContrastiveTask()` transform in the [DPO example](https://github.com/google-deepmind/gemma/tree/main/examples/dpo.py) for an example.

Here, we try [mtnt](https://www.tensorflow.org/datasets/catalog/mtnt), a small translation dataset. The dataset structure is `{'src': ..., 'dst': ...}`.

In [7]:
classification_labels = {
    'Topic': "Thread topic",
    'Development the previous message': "user add details to his own previous thought",
    "Question elaborating": "addition explanation of the topic question",
    'Off topic': "the message is not relevant to the topic of the conversation",
    'Found the new Idea': "The user suggests a new solution to the topic",
    'Found existing solution': "The user offers an existing solution to the topic with a link",
    # 'Duplicate',
    # 'Sensitive info'
    }
list_classes=[]
classes = [(i+1,key,value) for i, (key, value) in enumerate(classification_labels.items())]
for i, key, desc in classes:
    list_classes.append(f"{i}: '{key}' - {desc}")
list_classes

["1: 'Topic' - Thread topic",
 "2: 'Development the previous message' - user add details to his own previous thought",
 "3: 'Question elaborating' - addition explanation of the topic question",
 "4: 'Off topic' - the message is not relevant to the topic of the conversation",
 "5: 'Found the new Idea' - The user suggests a new solution to the topic",
 "6: 'Found existing solution' - The user offers an existing solution to the topic with a link"]

In [14]:
import numpy as np

text_classes = '\n '.join(list_classes)
system_prompt = f"""<start_of_turn>user
You are a helpful assistant that classifies messages of conversations in a blockhain forum.
For the message provided and its context, which is the preceding conversation,
identify the most appropriate label from the following list:\n {text_classes}.
Your response MUST be a number of the only predicted class label.
"""

user_prompt="""
{text}<end_of_turn>
<start_of_turn>model"""


dict_classes = {key: i for i, key, desc in classes}
tokenized_classes = {i: tokenizer.encode(key, add_bos=False)[0] for i, key, desc in classes}

def to_kauldron_dict(row):
  """Maps a dict row to a dictionary for Kauldron."""
  return {
      'inputs': '\n'.join([
          "Message:",
          row['message'],
          "Context:",
          row['context'][-2000:]
      ]),
      'label': dict_classes.get(row['predicted_class'])
  }
_INPUT_FIELD = "inputs"  # pylint: disable=invalid-name
_LABEL_FIELD = "label"  # pylint: disable=invalid-name

tokenizer = gm.text.Gemma3Tokenizer()
# Create the Kauldron dataset from the DataFrame
ds_classification = kd.data.py.Json(
    path = '/workspace/new labels/classification_results.json',
    batch_size=4, # Using a small batch size for demonstration
    shuffle=True,
    transforms=[
        # Apply the mapping function first
        to_kauldron_dict,
        # Process the input text
        gm.data.FormatText(
            key=_INPUT_FIELD,
            template='\n'.join([system_prompt,user_prompt]),
        ),
        gm.data.Tokenize(
            key=_INPUT_FIELD,
            tokenizer=tokenizer,
            add_bos=True,
        ),
        gm.data.Pad(
            key=_INPUT_FIELD,
            max_length=1024,
            truncate=True
        ),
        # Process the label
        # gm.data.MapInts(
        #     key=_LABEL_FIELD,
        #     # Rather than predicting the token 0 and 1, we are using the
        #     # token 1294 and 3553 which respectivelly correspond to "No" and
        #     # "Yes". We do this because those token already contain semantic
        #     # information, so even zero-shot prediction without any
        #     # finetuning has better than random performances.
        #     old_to_new=tokenized_classes,
        # ),
        kd.data.Rearrange(
            key=_LABEL_FIELD,
            pattern="... -> ... 1",  # For shape compatibility with the loss.
        ),


    ]
)

# Inspect an example from the dataset
ex_classification = next(iter(ds_classification))
treescope.show(ex_classification)

Disabling pygrain multi-processing (unsupported in colab).


{
  'inputs': # np.ndarray int64(4, 1024) [≥0, ≤247_998] zero:2_811 nonzero:1_285
    array([[   2,  105, 2364, ...,    0,    0,    0],
           [   2,  105, 2364, ...,    0,    0,    0],
           [   2,  105, 2364, ...,    0,    0,    0],
           [   2,  105, 2364, ...,    0,    0,    0]])
  ,
  'label': # np.ndarray int64(4, 1) [≥1, ≤3] nonzero:4
    array([[2],
           [3],
           [1],
           [3]])
  ,
}

We can decode an example from the batch to inspect the model input. We see that the `<start_of_turn>` / `<end_of_turn>` where correctly added to follow Gemma dialog format.

In [9]:
text = tokenizer.decode(ex_classification['inputs'][0])

print(text)

<start_of_turn>user
You are a helpful assistant that classifies messages of conversations in a blockhain forum.
For the message provided and its context, which is the preceding conversation,
identify the most appropriate label from the following list:
 1: 'Topic' - Thread topic
 2: 'Development the previous message' - user add details to his own previous thought
 3: 'Question elaborating' - addition explanation of the topic question
 4: 'Off topic' - the message is not relevant to the topic of the conversation
 5: 'Found the new Idea' - The user suggests a new solution to the topic
 6: 'Found existing solution' - The user offers an existing solution to the topic with a link.
Your response MUST be a number of the only predicted class label.


Message:
983307700836372480: is this the latest version - 24.12.99.8 (48d853fa) ?

Context:
834447372733906984:  Can i become a node runner or operator? what are the requirements
862550907349893151:  You will need 30m stake to join the validator
86

## Trainer

The [kauldron](https://kauldron.readthedocs.io/en/latest/) trainer allow to train Gemma simply by providing a dataset, a model, a loss and an optimizer.

Dataset, model and losses are connected together through a `key` strings system. For more information, see the [key documentation](https://kauldron.readthedocs.io/en/latest/intro.html#keys-and-context).

Each key starts by a registered prefix. Common prefixes includes:

* `batch`: The output of the dataset (after all transformations). Here our batch is `{'input': ..., 'loss_mask': ..., 'target': ...}`
* `preds`: The output of the model. For Gemma models, this is `gm.nn.Output(logits=..., cache=...)`
* `params`: Model parameters (can be used to add a weight decay loss, or monitor the params norm in metrics)






We then create the trainer:

In [27]:
trainer = kd.train.Trainer(
      seed=42,
      workdir='/workspace/new labels/LoRA_checkpoints',
      # Dataset
      train_ds=ds_classification,
      # Model definition
      model=gm.nn.LoRA(
          rank=4,
          model=gm.nn.Gemma3_1B(
              tokens=f"batch.{_INPUT_FIELD}",
              # TODO(epot): At the moment, LoRA fine-tuning with multimodal
              # is not supported. Willbe  fixed soon.
              return_last_only=True,
          )
      ),
      # model=gm.nn.Gemma3_1B(
      #     tokens=f"batch.{_INPUT_FIELD}",
      #     return_last_only=True,
      # ),
      # Load the weights from the pretrained checkpoint
      init_transform=gm.ckpts.SkipLoRA(
          wrapped=gm.ckpts.LoadCheckpoint(
              path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
          )
      ),
      # init_transform=gm.ckpts.LoadCheckpoint(
      #     path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
      # ),
      # Training
      num_train_steps=10000,
      train_losses={
          "xentropy": kd.losses.SoftmaxCrossEntropyWithIntLabels(
              logits="preds.logits",
              labels=f"batch.{_LABEL_FIELD}",
              mask="batch.loss_mask",
          ),
      },
      optimizer=kd.optim.partial_updates(
          optax.adafactor(learning_rate=0.005),#learning_rate=1e-4),
          # We only optimize the LoRA weights. The rest of the model is frozen.
          mask=kd.optim.select("lora"),
      ),      
      # optimizer=optax.adafactor(learning_rate=1e-4),
      checkpointer=kd.ckpts.Checkpointer(
          save_interval_steps=500,
      ),
      # # Evaluation
      # evals={
      #     "test": kd.evals.Evaluator(
      #         run=kd.evals.EveryNSteps(1000),
      #         ds=_make_dataset(training=False),
      #     ),
      # },
  )

Trainning can be launched with the `.train()` method.

Note that the trainer like the model are immutables, so it does not store the state nor params. Instead the state containing the trained parameters is returned.

In [ ]:
state, aux = trainer.train()

In [32]:
state.params.keys()

dict_keys(['embedder', 'final_norm', 'layer_0', 'layer_1', 'layer_10', 'layer_11', 'layer_12', 'layer_13', 'layer_14', 'layer_15', 'layer_16', 'layer_17', 'layer_18', 'layer_19', 'layer_2', 'layer_20', 'layer_21', 'layer_22', 'layer_23', 'layer_24', 'layer_25', 'layer_3', 'layer_4', 'layer_5', 'layer_6', 'layer_7', 'layer_8', 'layer_9'])

## Checkpointing

To save the model params, you can either:

* Activate checkpointing in the trainer by adding:

  ```python
  trainer = kd.train.Trainer(
      workdir='/tmp/my_experiment/',
      checkpointer=kd.ckpts.Checkpointer(
          save_interval_steps=500,
      ),
      ...
  )
  ```

  This will also save the optimizer, step, dataset state,...


* Manually save the trained params:

  ```python
  gm.ckpts.save_params(state.params, '/tmp/my_ckpt/')
  ```

## Evaluation

Here, we only perform a qualitative evaluation by sampling a prompt.

For more info on evals:

* See the [sampling](https://gemma-llm.readthedocs.io/en/latest/sampling.html) tutorial for more info on running inference.
* To add evals during training, see the Kauldron [evaluator](https://kauldron.readthedocs.io/en/latest/eval.html) documentation.


We test a sentence, using the same formatting used during fine-tuning:

In [34]:
from gemma import peft  # Parameter fine-tuning module
model = gm.nn.LoRA(
    rank=4,
    model=gm.nn.Gemma3_1B(),
)
original, lora = peft.split_params(state.params)
original = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT, params=original)
params = peft.merge_params(original, lora)

In [ ]:
peft.

In [17]:
import json
f = open('/workspace/new labels/classification_results.json', 'r')
data = json.load(f)
f.close()

In [35]:
import jax
import jax.numpy as jnp

import json
num_data=5

str_data = to_kauldron_dict(data[num_data])
print(data[num_data]["predicted_class"])
txt = gm.data.FormatText(
        key=_INPUT_FIELD,
        template='\n'.join([system_prompt,user_prompt])
    ).map({"inputs":str_data})['inputs']
prompt = tokenizer.encode(
    txt,
    add_bos=True)
prompt = jnp.asarray(prompt)
# Run the model

# model= gm.nn.Gemma3_1B(
#           tokens=f"batch.{_INPUT_FIELD}",
#       )
out = model.apply(
    {'params': params},
    tokens=prompt,
    return_last_only=True,  # Only predict the last token
)


# Sample a token from the predicted logits
next_token = jax.random.categorical(
    jax.random.key(1),
    out.logits
)
# {i: key for i, key, desc in classes}[int()]

Question elaborating


In [36]:
next_token

Array(5091, dtype=int32)

In [37]:
tokenizer.decode(next_token)

' please'

## Next steps

To fine-tune outside of Colab, you can look at our [examples/](https://github.com/google-deepmind/gemma/tree/main/examples/) folder for more complexes trainer configs, including LoRA, DPO and sharding.